In [ ]:
!pip install scikit-learn==1.9.1

In [ ]:
!ls /content

realty_data.csv  sample_data


**Чтение данных**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/realty_data.csv")

print(df.shape)
display(df.head())
df.info()

(98822, 17)


,product_name,period,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,settlement,district,area,description,source
0,"3-комнатная, 137 м²",NaN,63000000,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,NaN,Тверской район,NaN,Просторная квартира свободной планировки с пан...,ЦИАН
1,"Студия, 16,7 м²",NaN,3250000,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,NaN,Филимонковское поселение,NaN,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик
2,"3-комнатная, 76 м²",NaN,16004680,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,NaN,Сосенское поселение,NaN,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость
3,"1-комнатная, 24 м²",NaN,7841776,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,NaN,Сосенское поселение,NaN,Продается однокомнатная квартира № 381 в новос...,Новострой-М
4,"3-комнатная, 126 м²",NaN,120000000,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,NaN,Фили-Давыдково район,NaN,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98822 entries, 0 to 98821
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_name  98822 non-null  object 
 1   period        0 non-null      float64
 2   price         98822 non-null  int64  
 3   postcode      93675 non-null  float64
 4   address_name  98821 non-null  object 
 5   lat           98822 non-null  float64
 6   lon           98822 non-null  float64
 7   object_type   98822 non-null  object 
 8   total_square  98822 non-null  float64
 9   rooms         94840 non-null  float64
 10  floor         98822 non-null  float64
 11  city          91928 non-null  object 
 12  settlement    6894 non-null   object 
 13  district      75111 non-null  object 
 14  area          19498 non-null  object 
 15  description   98573 non-null  object 
 16  source        98822 non-null  object 
dtypes: float64(7), int64(1), object(9)
memory usage: 12.8+ MB


**Определение критериев для обучения**

In [ ]:
target = "price"
features = ["total_square", "rooms", "floor", "lat", "lon"]

df = df.dropna(subset=[target]).copy()

# Убедимся, что выбранные столбцы числовые
for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=[target])
X = df[features]
y = df[target]

print("Строк для обучения:", len(df))
print("Пропуски в признаках:\n", X.isna().sum())

Строк для обучения: 98822
Пропуски в признаках:
 total_square       0
rooms           3982
floor              0
lat                0
lon                0
dtype: int64


**Обучение модели**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, predictions), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, predictions)), 2))
print("R²:", round(r2_score(y_test, predictions), 3))

MAE: 2970422.97
RMSE: 10265469.59
R²: 0.921


In [ ]:
def predict_price(total_square, rooms, floor, lat, lon):
    sample = pd.DataFrame([{
        "total_square": total_square,
        "rooms": rooms,
        "floor": floor,
        "lat": lat,
        "lon": lon
    }])
    return model.predict(sample)[0]

example = predict_price(
    total_square=50,
    rooms=2,
    floor=5,
    lat=df["lat"].median(),
    lon=df["lon"].median()
)

print(f"Прогноз цены: {example:.0f}")

Прогноз цены: 31547794


**Создание приложения Streamlit**

In [ ]:
import os
import joblib

project_dir = "/content/realty_app"
os.makedirs(project_dir, exist_ok=True)

joblib.dump(model, f"{project_dir}/model.joblib")

# Сохраним диапазоны значений для полей интерфейса
settings = {
    "features": features,
    "minimums": X.min().to_dict(),
    "maximums": X.max().to_dict(),
    "medians": X.median().to_dict()
}

import json
with open(f"{project_dir}/settings.json", "w") as f:
    json.dump(settings, f)

In [ ]:
%%writefile /content/realty_app/app.py
import json
import joblib
import pandas as pd
import streamlit as st

st.title("Прогноз стоимости квартиры")
st.write("Введите характеристики квартиры.")

model = joblib.load("model.joblib")

with open("settings.json") as f:
    settings = json.load(f)

minimums = settings["minimums"]
maximums = settings["maximums"]
medians = settings["medians"]

total_square = st.number_input(
    "Общая площадь, м²",
    min_value=float(minimums["total_square"]),
    max_value=float(maximums["total_square"]),
    value=float(medians["total_square"])
)

rooms = st.number_input(
    "Количество комнат",
    min_value=float(minimums["rooms"]),
    max_value=float(maximums["rooms"]),
    value=float(medians["rooms"]),
    step=1.0
)

floor = st.number_input(
    "Этаж",
    min_value=float(minimums["floor"]),
    max_value=float(maximums["floor"]),
    value=float(medians["floor"]),
    step=1.0
)

lat = st.number_input(
    "Широта",
    min_value=float(minimums["lat"]),
    max_value=float(maximums["lat"]),
    value=float(medians["lat"]),
    format="%.6f"
)

lon = st.number_input(
    "Долгота",
    min_value=float(minimums["lon"]),
    max_value=float(maximums["lon"]),
    value=float(medians["lon"]),
    format="%.6f"
)

if st.button("Рассчитать стоимость"):
    data = pd.DataFrame([{
        "total_square": total_square,
        "rooms": rooms,
        "floor": floor,
        "lat": lat,
        "lon": lon
    }])

    price = model.predict(data)[0]
    st.success(f"Прогнозируемая стоимость: {price:.0f}")

Overwriting /content/realty_app/app.py


In [ ]:
!pip -q install streamlit pyngrok

**Проверка прогноза из консоли**

In [24]:
def predict_from_console():
    total_square = float(input("Площадь, м²: "))
    rooms = float(input("Количество комнат: "))
    floor = float(input("Этаж: "))
    lat = float(input("Широта: "))
    lon = float(input("Долгота: "))

    apartment = pd.DataFrame([{
        "total_square": total_square,
        "rooms": rooms,
        "floor": floor,
        "lat": lat,
        "lon": lon
    }])

    price = model.predict(apartment)[0]
    print(f"Прогнозируемая стоимость квартиры: {price: .2f}")

predict_from_console()

Площадь, м²: 53
Количество комнат: 3
Этаж: 5
Широта: 55.77
Долгота: 36.234
Прогнозируемая стоимость квартиры:  7640675.70
